In [25]:
import numpy as np 
from scipy.stats import multivariate_normal 
from scipy.spatial import KDTree 
from scipy.spatial.distance import cdist 

In [26]:
# coords are the observed coordinates 
# w is a draw of the spatial random effect 
# C is the true covariance matrix 
def make_synthetic_data(n=50, sigma2 = 1.0, ell=0.3, seed=76): 
    rng = np.random.default_rng(seed)
    coords = rng.uniform(0, 1, size = (n, 2)) 
    D = cdist(coords, coords) 
    C = sigma2 * np.exp(-D / ell) # exponential quadratic 
    C += 1e-6 * np.eye(n) # numerical stability 
    w = rng.multivariate_normal(np.zeros(n), C) 

    return coords, w, C

coords, w, C = make_synthetic_data() 

In [27]:
# order the coordinates by the first one 
order = np.argsort(coords[:, 0]) 
coords_sorted = coords[order] 
w_sorted = w[order]

In [28]:
# find nearest neighbors via KD tree 
# need to make sure that neighbors of i come from i-1, i-2, ..., 0 
# also need to account for cases where i < m 
def build_neighbor_array(coords, m): 
    n = len(coords) 
    neighbor_idx = np.full((n, m), -1) 

    for i in range(1, n): 
        predecessors = coords[:i] 
        tree = KDTree(predecessors)

        k = min(m, i) 
        _, idx = tree.query(coords[i], k=k) 
        neighbor_idx[i, :k] = idx 

    return neighbor_idx 

neighbor_idx = build_neighbor_array(coords=coords_sorted, m=10)

In [ ]:
# helper function 
def exp_cov(coords_a, coords_b, sigma2=1.0, ell=0.3): 
    D = cdist(coords_a, coords_b) 
    return sigma2 * np.exp(-D / ell) 

# compute the B and F matrices need for NNGP log-likelihood 
# formulas taken from the 2016 paper by Datta et al 
def compute_B_and_F(coords, neighbor_idx): 
    n = len(neighbor_idx)
    B = [None] * n 
    F = np.zeros(n)

    for i in range(n): 
        idx = neighbor_idx[i][neighbor_idx[i] != -1] 
        if len(idx) == 0: 
            B[i] = np.zeros(0) 
            F[i] = exp_cov(coords[[i]], coords[[i]]) 
            continue 
        
        C_ii = exp_cov(coords[[i]], coords[[i]]) # scalar 
        C_i_neighbor = exp_cov(coords[[i]], coords[idx]) # 1 x k 
        C_neighbor_neighbor = exp_cov(coords[idx], coords[idx]) # k x k 

        B[i] = np.linalg.solve(C_neighbor_neighbor.T, C_i_neighbor.T).T.ravel()
        F[i] = C_ii - (C_i_neighbor @ np.linalg.solve(C_neighbor_neighbor, C_i_neighbor.T)).item()

    return B, F

B, F = compute_B_and_F(coords=coords_sorted, neighbor_idx=neighbor_idx)

    

<positron-console-cell-29>:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
<positron-console-cell-29>:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)


In [30]:
# compute the log-likelihood of the NNGP 
def compute_log_lik(w, neighbor_idx, B, F): 
    log_lik = 0 
    
    for i in range(len(w)): 
        idx = neighbor_idx[i][neighbor_idx[i] != -1]
        if len(idx) == 0: 
            term = -0.5 * np.log(2*np.pi) - 0.5 * np.log(F[i]) - 0.5 * np.square(w[i]) / F[i]
            log_lik += term 
            continue 

        resid = w[i] - B[i] @ w[idx] 
        term = -0.5 * np.log(2*np.pi) - 0.5 * np.log(F[i]) - 0.5 * np.square(resid) / F[i] 
        log_lik += term 

    return log_lik

log_lik = compute_log_lik(w=w_sorted, neighbor_idx=neighbor_idx, B=B, F=F)
print(log_lik)

-45.77543321752609


In [31]:
# validate against full GP log-likelihood 
# with m = 10, we got extremely close! everything seems to be working 
full_log_lik = multivariate_normal.logpdf(w, mean=np.zeros(len(w)), cov=C) 
print(full_log_lik)

-45.77832146580964
